In [0]:
df = spark.read.table("ecommerce_analytics.bronze.sales")
df.display()

In [0]:
{"curr":"USD","id":"AVpiE9hhilAPnD_xAfSU","name":"Cyber-shot DSC-RX100 V Digital Camera","price":2798,"qty":4,"unit":"pcs"}

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col

product_schema = ArrayType(
    StructType([
        StructField("curr", StringType()),
        StructField("id", StringType()),
        StructField("name", StringType()),
        StructField("price", StringType()),
        StructField("qty", StringType()),
        StructField("unit", StringType())
    ])
)

df_parsed_product = df.withColumn("product", from_json(col("product"), product_schema))
df_parsed_product.display()


In [0]:
from pyspark.sql.functions import explode_outer

df_explode_product = df_parsed_product.withColumn("product", explode_outer("product")).filter(col("product").isNotNull())
df_explode_product.display()

In [0]:
df_explode_product.printSchema()

In [0]:
df_product = df_explode_product.select(
    "customer_id",
    "customer_name",
    "order_date",
    "product_name",
    "product_category",
    col("product.id").alias("product_id"),
    col("product.name").alias("product_description"),
    col("product.price").alias("price"),
    col("product.qty").alias("qty"),
    col("product.unit").alias("unit"),
    "total_price",
    col("product.curr").alias("curr")
)
df_product.display()
